In [ ]:
import sqlite3
import os
import sys
import time

# ===============================
# パス設定（スクレイピングと共通）
# ===============================
try:
    base_dir = os.path.dirname(__file__)
except NameError:
    base_dir = os.getcwd()

sys.path.append(os.path.abspath(os.path.join(base_dir, "..")))
from utils.config import PROJECT_DIR

start_time = time.time()

user_base = os.path.join(
    os.environ["USERPROFILE"] if os.name == "nt" else os.path.expanduser("~"),
    "myenv310",
    PROJECT_DIR
)

# 移動先DB
target_db = os.path.join(user_base, "db", "data.db")

# 元DB
source_db = os.path.join(user_base, "db", "output.db")

print(f"[INFO] Source DB : {source_db}")
print(f"[INFO] Target DB : {target_db}")

conn = sqlite3.connect(target_db)
cur = conn.cursor()

try:
    cur.execute("ATTACH DATABASE ? AS src", (source_db,))

    cur.execute("""
        INSERT OR IGNORE INTO result_table (
            sku,
            shop_name,
            shop_product_id,
            scraped_date,
            created_at,
            updated_at,
            machine_name,
            normalized_machine_name,
            price,
            product_url,
            image_url,
            stock,
            maker,
            category,
            machine_category,
            series,
            machine_series,
            master_machine_id,
            master_machine_pworld_url,
            master_machine_pworldimage_url,
            master_machine_name,
            master_machine_model_TEXT,
            master_machine_maker,
            master_machine_type,
            master_machine_gouki,
            master_machine_memo,
            master_machine_tag,
            master_machine_dis,
            status,
            remarks
        )
        SELECT
            sku,
            shop_name,
            shop_product_id,
            scraped_date,
            created_at,
            updated_at,
            machine_name,
            normalized_machine_name,
            price,
            product_url,
            image_url,
            stock,
            maker,
            category,
            machine_category,
            series,
            machine_series,
            master_machine_id,
            master_machine_pworld_url,
            master_machine_pworldimage_url,
            master_machine_name,
            master_machine_model_TEXT,
            master_machine_maker,
            master_machine_type,
            master_machine_gouki,
            master_machine_memo,
            master_machine_tag,
            master_machine_dis,
            status,
            remarks
        FROM src.result_table
    """)

    conn.commit()

    print(f"[INFO] コピー完了")
    print(f"[INFO] 追加件数: {cur.rowcount}")

finally:
    cur.execute("DETACH DATABASE src")
    conn.close()

print(f"[INFO] 完了 ({time.time() - start_time:.2f}秒)")